In [2]:
from pyspark.sql import SparkSession
import time
spark = SparkSession.builder.appName("DECrossSkiling").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/14 12:12:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.types import StructType,StructField,StringType, IntegerType, DoubleType, TimestampNTZType

schema = StructType([
    StructField("Unique Key", IntegerType(), nullable=False),
    StructField("Created Date", TimestampNTZType(), nullable=True),
    StructField("Closed Date", TimestampNTZType(), nullable=True),
    StructField("Agency", StringType(), nullable=True),
    StructField("Agency Name", StringType(), nullable=True),
    StructField("Complaint Type", StringType(), nullable=True),
    StructField("Descriptor", StringType(), nullable=True),
    StructField("Location Type", StringType(), nullable=True),
    StructField("Incident Zip", StringType(), nullable=True),
    StructField("Incident Address", StringType(), nullable=True),
    StructField("Street Name", StringType(), nullable=True),
    StructField("Cross Street 1", StringType(), nullable=True),
    StructField("Cross Street 2", StringType(), nullable=True),
    StructField("Intersection Street 1", StringType(), nullable=True),
    StructField("Intersection Street 2", StringType(), nullable=True),
    StructField("Address Type", StringType(), nullable=True),
    StructField("City", StringType(), nullable=True),
    StructField("Landmark", StringType(), nullable=True),
    StructField("Facility Type", StringType(), nullable=True),
    StructField("Status", StringType(), nullable=True),
    StructField("Due Date", StringType(), nullable=True),
    StructField("Resolution Description", StringType(), nullable=True),
    StructField("Resolution Action Updated Date", StringType(), nullable=True),
    StructField("Community Board", StringType(), nullable=True),
    StructField("BBL", StringType(), nullable=True),
    StructField("Borough", StringType(), nullable=True),
    StructField("X Coordinate (State Plane)", StringType(), nullable=True),
    StructField("Y Coordinate (State Plane)", StringType(), nullable=True),
    StructField("Open Data Channel Type", StringType(), nullable=True),
    StructField("Park Facility Name", StringType(), nullable=True),
    StructField("Park Borough", StringType(), nullable=True),
    StructField("Vehicle Type", StringType(), nullable=True),
    StructField("Taxi Company Borough", StringType(), nullable=True),
    StructField("Taxi Pick Up Location", StringType(), nullable=True),
    StructField("Bridge Highway Name", StringType(), nullable=True),
    StructField("Bridge Highway Direction", StringType(), nullable=True),
    StructField("Road Ramp", StringType(), nullable=True),
    StructField("Bridge Highway Segment", StringType(), nullable=True),
    StructField("Latitude", StringType(), nullable=True),
    StructField("Longitude", StringType(), nullable=True),
    StructField("Location", StringType(), nullable=True),
    StructField("Zip Codes", StringType(), nullable=True),
    StructField("Community Districts", StringType(), nullable=True),
    StructField("Borough Boundaries", StringType(), nullable=True),
    StructField("City Council Districts", StringType(), nullable=True),
    StructField("Police Precincts", StringType(), nullable=True)

])

In [4]:
service_requests_df_json = (spark.read
                            # .schema(schema)
                           .json("../data/archive/socrata_metadata_311-service-requests-from-2010-to-present.json"))

26/10/14 12:12:43 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [5]:
service_requests_df_csv = (spark.read
                            # .schema(schema)
                           .csv("../data/archive/311-service-requests-from-2010-to-present.csv"))

In [6]:
service_requests_df_parquet = (spark.read
                            # .schema(schema)
                           .parquet("../data/archive/311-service-requests-from-2010-to-present.parquet/"))

In [7]:
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql import DataFrame

# service_requests_df_csv_year = service_requests_df_csv.withColumn("year", F.year(F.col("created_date")))
# service_requests_df_json_year = service_requests_df_json.withColumn("year", F.year(F.col("created_date")))
# service_requests_df_parquet_year = service_requests_df_parquet.withColumn("year", F.year(F.col("created_date")))

combined = service_requests_df_parquet.unionByName(service_requests_df_csv, allowMissingColumns=True)
combined_df = combined.unionByName(service_requests_df_json, allowMissingColumns=True)


In [8]:
combined_df_ren = (combined_df
                              .withColumnRenamed("Unique Key", "unique_key")
                              .withColumnRenamed("Created Date", "created_date")
                              .withColumnRenamed("Closed Date", "closed_date")
                              .withColumnRenamed("Complaint Type", "complaint_type")
                              .withColumnRenamed("Borough", "borough")
                              .withColumnRenamed("Agency", "agency")
                              .withColumnRenamed("Agency Name", "agency_name")
                              .withColumnRenamed("Status", "status")
                              .withColumnRenamed("Descriptor", "descriptor")
                              .withColumnRenamed("Location Type", "location_type")
                              .withColumnRenamed("Incident Zip", "incident_zip")
                              .withColumnRenamed("Incident Address", "incident_address")
                              .withColumnRenamed("Street Name", "street_name")
                              .withColumnRenamed("Cross Street 1", "cross_street_1")
                              .withColumnRenamed("Cross Street 2", "cross_street_2")
                              .withColumnRenamed("Intersection Street 1", "intersection_street_1")
                              .withColumnRenamed("Intersection Street 2", "intersection_street_2")
                              .withColumnRenamed("Address Type", "address_type")
                              .withColumnRenamed("City", "city")
                              .withColumnRenamed("Landmark", "landmark")
                              .withColumnRenamed("Facility Type", "facility_type")
                              .withColumnRenamed("Due Date", "due_date")
                              .withColumnRenamed("Resolution Description", "resolution_description")
                              .withColumnRenamed("Resolution Action Updated Date", "resolution_action_updated_date")
                              .withColumnRenamed("Community Board", "community_board")
                              .withColumnRenamed("BBL", "bbl")
                              .withColumnRenamed("X Coordinate (State Plane)", "x_coordinate_state_plane")
                              .withColumnRenamed("Y Coordinate (State Plane)", "y_coordinate_state_plane")
                              .withColumnRenamed("Open Data Channel Type", "open_data_channel_type")
                              .withColumnRenamed("Park Facility Name", "park_facility_name")
                              .withColumnRenamed("Park Borough", "park_borough")
                              .withColumnRenamed("Vehicle Type", "vehicle_type")
                              .withColumnRenamed("Taxi Company Borough", "taxi_company_borough")
                              .withColumnRenamed("Taxi Pick Up Location", "taxi_pick_up_location")
                              .withColumnRenamed("Bridge Highway Name", "bridge_highway_name")
                              .withColumnRenamed("Bridge Highway Direction", "bridge_highway_direction")
                              .withColumnRenamed("Road Ramp", "road_ramp")
                              .withColumnRenamed("Bridge Highway Segment", "bridge_highway_segment")
                              .withColumnRenamed("Latitude", "latitude")
                              .withColumnRenamed("Longitude", "longitude")
                              .withColumnRenamed("Location", "location")
                              .withColumnRenamed("Zip Codes", "zip_codes")
                              .withColumnRenamed("Community Districts", "community_districts")
                              .withColumnRenamed("Borough Boundaries", "borough_boundaries")
                              .withColumnRenamed("City Council Districts", "city_council_districts")
                              .withColumnRenamed("Police Precincts", "police_precincts")
                              )

In [9]:
combined_df_ren.show(5)

26/10/14 12:13:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:>                                                          (0 + 1) / 1]

+----------+-------------------+-------------------+------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+----------------+----------------+--------------+---------------------+---------------------+------------+---------+----------------+-------------+-----------+--------+----------------------+------------------------------+---------------+----------+--------+------------------------+------------------------+----------------------+------------------+------------+------------+--------------------+---------------------+-------------------+------------------------+---------+----------------------+-----------------+------------------+--------------------+---------------+-------------------+------------------+----------------------+----------------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+---

In [11]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

null_count_threshold = 80  # percent
total_row = combined_df_ren.count()

# One aggregate per column: number of nulls
null_counts = combined_df_ren.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in combined_df_ren.columns
])

# Convert the single result row to a dict, then compute null percentages
null_row = null_counts.first().asDict()
null_pct = {c: (n / total_row) * 100 for c, n in null_row.items()}

# Columns whose null percentage exceeds the threshold
cols_to_drop = [c for c, pct in null_pct.items() if pct > null_count_threshold]

In [12]:
cols_to_drop

['address_type',
 'facility_type',
 'due_date',
 'vehicle_type',
 'taxi_company_borough',
 'taxi_pick_up_location',
 'bridge_highway_name',
 'bridge_highway_direction',
 'road_ramp',
 'bridge_highway_segment',
 '_c15',
 '_c18',
 '_c20',
 '_c31',
 '_c32',
 '_c33',
 '_c34',
 '_c35',
 '_c36',
 '_c37',
 'approvals',
 'attribution',
 'averageRating',
 'category',
 'columns',
 'createdAt',
 'description',
 'displayType',
 'downloadCount',
 'flags',
 'grants',
 'hideFromCatalog',
 'hideFromDataJson',
 'id',
 'indexUpdatedAt',
 'metadata',
 'name',
 'newBackend',
 'numberOfComments',
 'oid',
 'owner',
 'provenance',
 'publicationAppendEnabled',
 'publicationDate',
 'publicationGroup',
 'publicationStage',
 'query',
 'rights',
 'rowClass',
 'rowIdentifierColumnId',
 'rowsUpdatedAt',
 'rowsUpdatedBy',
 'tableAuthor',
 'tableId',
 'tags',
 'totalTimesRated',
 'viewCount',
 'viewLastModified',
 'viewType']

In [13]:
dropped_coln_dff = combined_df_ren.drop(*cols_to_drop)

In [14]:
dropped_coln_dff.printSchema()

root
 |-- unique_key: integer (nullable = true)
 |-- created_date: timestamp_ntz (nullable = true)
 |-- closed_date: timestamp_ntz (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- incident_zip: string (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- cross_street_1: string (nullable = true)
 |-- cross_street_2: string (nullable = true)
 |-- intersection_street_1: string (nullable = true)
 |-- intersection_street_2: string (nullable = true)
 |-- city: string (nullable = true)
 |-- landmark: string (nullable = true)
 |-- status: string (nullable = true)
 |-- resolution_description: string (nullable = true)
 |-- resolution_action_updated_date: string (nullable = true)
 |-- community_board: string (nullable = true)
 |-- bbl: string (nullable 

In [15]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col


SLA_HOURS = 72 

df_sla_fin = (
    dropped_coln_dff
    # 1. Resolution time in hours (null while closed_date is null)
    .withColumn("is_open", col("closed_date").isNull())
    .withColumn(
        "effective_closed_date",
        F.coalesce(col("closed_date"), F.current_timestamp())
    )

    # 1. Resolution time in hours (open requests = elapsed time so far)
    .withColumn(
        "resolution_hours",
        (F.unix_timestamp(col("effective_closed_date")) - F.unix_timestamp(col("created_date"))) / 3600
    )

    # 2. Within SLA? (null only if the data is invalid)
    .withColumn(
        "within_sla",
        F.when(col("resolution_hours").isNull() | (col("resolution_hours") < 0), F.lit(None).cast("boolean"))
         .when(col("resolution_hours") <= F.lit(SLA_HOURS), F.lit(True))
         .otherwise(F.lit(False))
    )
)

In [16]:
df_sla_fin.show(4)

+----------+-------------------+-------------------+------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+----------------+----------------+--------------+---------------------+---------------------+---------+----------------+-----------+----------------------+------------------------------+---------------+----------+--------+------------------------+------------------------+----------------------+------------------+------------+-----------------+------------------+--------------------+---------------+-------------------+------------------+----------------------+----------------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-------+---------------------+------------------+----------+
|unique_key|       created_date|        closed_date|agency|         agency_name|      complaint_type| 

In [18]:
dropped_dup = df_sla_fin.dropDuplicates(['unique_key'])

In [19]:
desc_re_hr_df = dropped_dup.orderBy("resolution_hours", ascending=False)